In [77]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/agents-intensive-capstone-project/Hackathon dataset.txt


In [78]:
# Install Google ADK 
%pip install google-adk

Note: you may need to restart the kernel to use updated packages.


In [79]:
from kaggle_secrets import UserSecretsClient
import os

# Fetch Gemini API key from Kaggle secrets
try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(f"🔑 Authentication Error: Please add 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}")

✅ Gemini API key setup complete.


In [80]:
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
import asyncio

In [81]:
# Retry config for HTTP requests
retry_config = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],
)

# Create folder + write tools.py

In [83]:
import os

# Create the agent folder
os.makedirs("fraud_detector_agent", exist_ok=True)

# Write tools.py file
tools_code = """
from google.genai import Client
import os

# Create shared Gemini client
client = Client(api_key=os.environ["GOOGLE_API_KEY"])

def analyze_image(image_bytes: bytes) -> str:
    result = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=[
            {"mime_type": "image/jpeg", "data": image_bytes},
            {"text": "Analyze the product. Identify brand, model, authenticity clues."}
        ]
    )
    return result.text

def test_tool(message: str) -> str:
    return f'Tool received: {message}'
"""

with open("fraud_detector_agent/tools.py", "w") as f:
    f.write(tools_code)

print("✅ tools.py created successfully!")

✅ tools.py created successfully!


# Create agent_local.py

In [84]:
agent_local_code = """
import os
from google.genai import Client
from fraud_detector_agent.tools import analyze_image, test_tool

class FraudDetectorAgent:

    def __init__(self):
        self.client = Client(api_key=os.environ["GOOGLE_API_KEY"])

    def run(self, message: str, image_bytes=None):
        if image_bytes:
            return analyze_image(image_bytes)

        result = self.client.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=message
        )
        return result.text

    def debug(self, msg):
        return test_tool(msg)

"""

with open("fraud_detector_agent/agent_local.py", "w") as f:
    f.write(agent_local_code)


print("✅ agent_local.py created!")

✅ agent_local.py created!


# Import & verify

In [85]:
import fraud_detector_agent.tools as t
import fraud_detector_agent.agent_local as a

print("Tools found:", dir(t))

Tools found: ['Client', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'analyze_image', 'client', 'os', 'test_tool']


# Initialize agent

In [86]:
from fraud_detector_agent.agent_local import FraudDetectorAgent
agent = FraudDetectorAgent()

print("Agent initialized!")

Agent initialized!


# Test the agent

In [87]:
print(agent.run("Is the agent working?"))

I am a large language model, trained by Google.


# Tool test

In [88]:
print(agent.debug("Hello Tool"))

Tool received: Hello Tool


In [89]:
# Core libraries
import os
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.genai import types

# Retry configuration for API requests
retry_config = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],
)

In [90]:
# Text analysis tools for the agent
# Load environment variables safely
from dotenv import load_dotenv
import os

# Load .env file (make sure .env is in your project folder)
load_dotenv('/kaggle/working/fraud_detector_agent/.env')

# Google Gemini client
from google.genai import Client

# Optional debug tool
def test_tool(message: str) -> str:
    return f"Tool received: {message}"

# Fraud Detector Agent class
class FraudDetectorAgent:
    def __init__(self):
        # Initialize client with API key from .env
        self.client = Client(api_key=os.environ["GOOGLE_API_KEY"])

    def run(self, message: str) -> str:
        # Main text analysis
        result = self.client.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=message
        )
        return result.text

    def debug(self, msg: str) -> str:
        # Tool for testing/debug
        return test_tool(msg)

# Instantiate agent
agent = FraudDetectorAgent()

In [91]:
# Test text analysis
output = agent.run("Is this product description suspicious?")
print(output)

# Test the debug tool
debug_output = agent.debug("Hello Tool")
print(debug_output)

To help me determine if a product description is suspicious, I need you to **provide me with the product description itself.**

Once you share it, I'll look for common red flags and tell you my assessment.

**What I'll be looking for includes things like:**

*   **Unbelievable claims or prices:** "Guaranteed to make you a millionaire overnight!" or prices that are drastically lower than comparable products.
*   **Vague or generic language:** Lack of specific details about the product, its features, or its origin.
*   **Poor grammar and spelling:** While not always a sign of a scam, it can be a warning sign, especially if it's pervasive.
*   **Pressure tactics:** Urgency to buy immediately ("Limited time offer! Only 2 left!").
*   **Request for unusual payment methods:** Asking for payment via gift cards, wire transfers, or cryptocurrency for a legitimate product.
*   **Lack of contact information or a physical address:** A legitimate seller will usually provide ways to contact them.
* 

# Test with a sample product description

In [92]:
# Example product description
description = """
This amazing weight loss supplement will make you lose 10 pounds in a week!
Guaranteed results with no effort. Limited stock, buy now!
"""

# Run the agent on the description
output = agent.run(description)
print(output)

This advertisement is using several persuasive techniques, but it's important to be aware of what they are and whether they're backed by evidence. Let's break it down:

**Persuasive Techniques Used:**

*   **Bold Claims/Exaggeration:** "lose 10 pounds in a week!" and "Guaranteed results with no effort" are extremely bold claims. Sustainable and healthy weight loss typically involves gradual changes. Such rapid loss is often unhealthy and unsustainable.
*   **Sense of Urgency:** "Limited stock, buy now!" creates a fear of missing out (FOMO) and pressures the consumer to make a quick decision without proper consideration.
*   **Promise of Effortless Solution:** "no effort" is a highly attractive but often unrealistic promise. Weight loss generally requires some level of lifestyle change, including diet and/or exercise.
*   **Implied Authority/Trustworthiness:** While not explicitly stated, the confident and declarative tone suggests the product is reliable and effective.

**Why You Shoul

# Test with another description

In [93]:
# Another sample description
description = """
This smartwatch tracks your heart rate, steps, and sleep patterns.
Available in multiple colors. Limited time offer!
"""

# Run the agent on the description
output = agent.run(description)
print(output)

This is a great, concise product description! It highlights the key features and benefits quickly.

Here are a few ways you could build on this, depending on where you're using it and what your goals are:

**For a Short Product Listing (e.g., social media ad, brief product name):**

*   **Smartwatch: Heart Rate, Steps, Sleep. Multiple Colors. Limited Time Offer!** (Very direct)
*   **Track Your Health: Smartwatch with Heart Rate, Steps & Sleep. Many Colors. Sale Ends Soon!** (Focuses on benefit)

**For a Slightly Longer Description (e.g., website product page, email):**

*   **Stay on top of your fitness and well-being with our advanced smartwatch. It accurately tracks your heart rate, counts your daily steps, and monitors your sleep patterns, giving you valuable insights into your health. Choose from a range of stylish colors to match your personal style. Don't miss out – this is a limited-time offer!** (Adds a bit more descriptive language)
*   **Introducing the ultimate companion fo

# Interactive input for product descriptions

In [59]:
# Simple interactive input for Kaggle notebook
description = input("Paste the product description here:\n")

# Run the agent
output = agent.run(description)
print("\n--- Agent Assessment ---\n")
print(output)

Paste the product description here:
 Lose 10 pounds in a week with no exercise or diet changes



--- Agent Assessment ---

It's **not realistically possible or healthy to lose 10 pounds in a week without any exercise or diet changes.**

Here's why:

*   **Healthy Weight Loss:** Sustainable and healthy weight loss is generally considered to be 1-2 pounds per week. This is achieved through a calorie deficit created by a combination of diet and exercise.
*   **Calorie Deficit:** To lose 1 pound of fat, you need to create a deficit of approximately 3,500 calories. To lose 10 pounds of fat, you would need a deficit of 35,000 calories. This is an enormous amount to achieve in just seven days without making any changes to what you eat or how much you move.
*   **Water Weight vs. Fat Loss:** The only way to experience a significant weight drop in such a short period without changes is through a loss of water weight. This can happen due to:
    *   **Dehydration:** While you might see a lower number on the scale, this is extremely unhealthy and dangerous. Severe dehydration can lead to se

In [60]:
# Simple interactive input for Kaggle notebook
description = input("Paste the product description here:\n")

# Run the agent
output = agent.run(description)
print("\n--- Agent Assessment ---\n")
print(output)

Paste the product description here:
 High-quality stainless steel water bottle, 500ml, BPA-free.



--- Agent Assessment ---

Here are a few options for a high-quality stainless steel water bottle that fit your criteria (500ml, BPA-free), along with some considerations for choosing the best one for you:

**Top Picks & What to Look For:**

When looking for a "high-quality" stainless steel bottle, you're generally looking for:

*   **Food-Grade Stainless Steel (18/8 or 304):** This is the industry standard for safe and durable stainless steel. It's rust-resistant, doesn't impart flavors, and is non-toxic.
*   **Double-Wall Vacuum Insulation:** This is crucial for keeping your drinks hot or cold for extended periods. It also prevents condensation from forming on the outside.
*   **Leak-Proof Lid:** Essential for carrying it in a bag.
*   **Durable Finish:** A good powder coating or other finish will resist scratches and maintain its appearance.
*   **Ease of Cleaning:** Wide mouth openings are generally easier to clean and add ice to.
*   **Good Grip:** Some bottles have a textured fin

In [94]:
# Function to assess multiple product descriptions
def assess_products(descriptions):
    results = {}
    for i, desc in enumerate(descriptions, start=1):
        assessment = agent.run(desc)
        results[f"Product {i}"] = assessment
    return results

# Example usage
sample_descriptions = [
    "Lose 10 pounds in a week with no exercise or diet changes!",
    "High-quality stainless steel water bottle, 500ml, BPA-free.",
    "Get rich fast with this secret online method, guaranteed income in 7 days!"
]

assessments = assess_products(sample_descriptions)

# Display results
for product, result in assessments.items():
    print(f"{product} Assessment:\n{result}\n{'-'*50}")

Product 1 Assessment:
Losing 10 pounds in a week without any exercise or diet changes is **not a realistic or healthy goal.**

Here's why and what might contribute to a perceived, but temporary, weight change:

**Why it's not realistic:**

*   **Fat Loss vs. Water Loss:** Significant fat loss takes time. To lose 10 pounds of pure fat, you'd need to be in a calorie deficit of around 35,000 calories (since 1 pound of fat is roughly 3500 calories). Achieving this in a week without drastic measures is impossible.
*   **Health Risks:** Any method that claims to achieve this kind of rapid weight loss without lifestyle changes is likely to involve unhealthy practices, dehydration, or extreme calorie restriction, all of which can be dangerous.

**What might *seem* like weight loss but isn't sustainable or healthy:**

The only way to see a significant drop on the scale this quickly without changing what you eat or how much you move is usually due to **water loss and waste elimination.** This ca

# Import IPython widgets

In [96]:
from IPython.display import display, clear_output
import ipywidgets as widgets

# Connect button to agent and display output

In [101]:
import ipywidgets as widgets
from IPython.display import display

# Callback function when button is clicked
def on_analyze_clicked(b):
    with output_area:
        output_area.clear_output()  # Clear previous results
        description = description_input.value
        if description.strip() == "":
            print("⚠️ Please enter a product description.")
        else:
            # Run the agent on the input description
            result = agent.run(description)
            print(result)

# Link button click to callback
analyze_button.on_click(on_analyze_clicked)

# Create input box, button, and output area

In [108]:
# Input box for product description
import ipywidgets as widgets
from IPython.display import display

# Callback function when button is clicked
def on_analyze_clicked(b):
    with output_area:
        output_area.clear_output()  # Clear previous results
        description = description_input.value
        if description.strip() == "":
            print("⚠️ Please enter a product description.")
        else:
            # Run the agent on the input description
            result = agent.run(description)
            print(result)

# Link button click to callback
analyze_button.on_click(on_analyze_clicked)

In [111]:
# Input box for product description
description_input = widgets.Textarea(
    value='',
    placeholder='Paste the product description here...',
    description='Product:',
    layout=widgets.Layout(width='70%', height='100px')
)

# Button to submit description
analyze_button = widgets.Button(
    description='Analyze Description',
    button_style='success'
)

# Output area to show agent response
output_area = widgets.Output()

display(description_input, analyze_button, output_area)

Textarea(value='', description='Product:', layout=Layout(height='100px', width='70%'), placeholder='Paste the …

Button(button_style='success', description='Analyze Description', style=ButtonStyle())

Output()

In [110]:
import ipywidgets as widgets
from IPython.display import display

# Callback function when button is clicked
def on_analyze_clicked(b):
    with output_area:
        output_area.clear_output()  # Clear previous results
        description = description_input.value
        if description.strip() == "":
            print("⚠️ Please enter a product description.")
        else:
            # Run the agent on the input description
            result = agent.run(description)
            print(result)

# Link button click to callback
analyze_button.on_click(on_analyze_clicked)

In [65]:
description = "Lose 20 pounds in 5 days with this magic diet pill. No exercise required! Limited time offer, only $9.99!"
print(agent.run(description))

I cannot endorse or promote any product or service that makes unsubstantiated health claims, such as promising significant weight loss in a very short period without exercise. These types of claims are often misleading and can be harmful.

**It's important to be aware of the following:**

*   **Unrealistic Weight Loss:** Losing 20 pounds in 5 days is an extreme and unhealthy amount of weight to lose. Sustainable and healthy weight loss typically occurs at a rate of 1-2 pounds per week. Rapid weight loss is often due to water loss and can lead to muscle loss, dehydration, electrolyte imbalances, and other health complications.
*   **"Magic" Pills:** There is no such thing as a "magic" diet pill that can cause rapid and effortless weight loss. Many such products are ineffective, and some can even be dangerous and contain undisclosed or harmful ingredients.
*   **Health Risks:** Relying on pills for weight loss without proper medical guidance can be detrimental to your health. It can mask

# Initialize Git

In [68]:
%%bash
cd /kaggle/working/fraud_detector_agent
git init
git config user.name "debashish967"
git config user.email "bora.debashish1@gmail.com"

Reinitialized existing Git repository in /kaggle/working/fraud_detector_agent/.git/


In [112]:
from fraud_detector_agent.agent_local import FraudDetectorAgent